In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Intro to Generating and Executing Python Code with Gemini 3

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fcode-execution%2Fintro_code_execution.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/code-execution/intro_code_execution.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>

| Author |
| --- |
| [Kristopher Overholt](https://github.com/koverholt/) |

## Overview

This notebook introduces the code execution capabilities of the [Gemini 3 Flash model](https://cloud.google.com/vertex-ai/generative-ai/docs/gemini-v2), a new multimodal generative AI model from Google [DeepMind](https://deepmind.google/). Gemini 3 Flash offers improvements in speed, quality, and advanced reasoning capabilities including enhanced understanding, coding, and instruction following.

## Code Execution

A key feature of this model is [code execution](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/code-execution), which is the ability to generate and execute Python code directly within the API. If you want the API to generate and run Python code and return the results, you can use code execution as demonstrated in this notebook.

This code execution capability enables the model to generate code, execute and observe the results, correct the code if needed, and learn iteratively from the results until it produces a final output. This is particularly useful for applications that involve code-based reasoning such as solving mathematical equations or processing text.

## Objectives

In this tutorial, you will learn how to generate and execute code using the Gemini API in Vertex AI and the Google Gen AI SDK for Python with the Gemini 3 Flash model.

You will complete the following tasks:

- Generating and running sample Python code from text prompts
- Exploring data using code execution in multi-turn chats
- Using code execution in streaming sessions

## Getting started

### Install Google Gen AI SDK for Python


In [1]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [3]:
import os

from IPython.display import Markdown, display
from google import genai
from google.genai.types import GenerateContentConfig, Tool, ToolCodeExecution

### Connect to a generative AI API service

Google Gen AI APIs and models including Gemini are available in the following two API services:

- [Google AI for Developers](https://ai.google.dev/gemini-api/docs): Experiment, prototype, and deploy small projects.
- [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs): Build enterprise-ready projects on Google Cloud.
The Google Gen AI SDK provides a unified interface to these two API services.

This notebook shows how to use the Google Gen AI SDK with the Gemini API in Vertex AI.

### Set Google Cloud project information and create client

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [4]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

In [5]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## Working with code execution in Gemini 3

### Load the Gemini model

The following code loads the Gemini 3 Flash model. You can learn about all Gemini models on Vertex AI by visiting the [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models):

In [6]:
MODEL_ID = "gemini-3-flash-preview"  # @param {type: "string"}

### Define the code execution tool

The following code initializes the code execution tool by passing `code_execution` in a `Tool` definition.

Later we'll register this tool with the model that it can use to generate and run Python code:

In [7]:
code_execution_tool = Tool(code_execution=ToolCodeExecution())

### Generate and execute code

The following code sends a prompt to the Gemini model, asking it to generate and execute Python code to calculate the sum of the first 50 prime numbers. The code execution tool is passed in so the model can generate and run the code:

In [8]:
PROMPT = """What is the sum of the first 50 prime numbers?
Generate and run code for the calculation."""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[code_execution_tool],
    ),
)

### View the generated code

The following code iterates through the response and displays any generated Python code by checking for `part.executable_code` in the response parts:

In [9]:
for part in response.candidates[0].content.parts:
    if part.executable_code:
        display(
            Markdown(
                f"""
```py
{part.executable_code.code}
```
"""
            )
        )


```py
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

primes = []
num = 2
while len(primes) < 50:
    if is_prime(num):
        primes.append(num)
    num += 1

sum_primes = sum(primes)
print(f"The first 50 prime numbers are: {primes}")
print(f"The sum of the first 50 prime numbers is: {sum_primes}")
```


### View the code execution results

The following code iterates through the response and displays the execution result and outcome by checking for `part.code_execution_result` in the response parts:

In [10]:
for part in response.candidates[0].content.parts:
    if part.code_execution_result:
        display(Markdown(f"`{part.code_execution_result.output}`"))
        print("\nOutcome:", part.code_execution_result.outcome)

`The first 50 prime numbers are: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229]
The sum of the first 50 prime numbers is: 5117
`


Outcome: Outcome.OUTCOME_OK


Great! Now you have the answer (`5117`) as well as the generated (and verified via execution!) Python code.

At this point in your application, you would save the output code, result, or outcome and display it to the end-user or use it downstream in your application.

### Code execution in a chat session

This section shows how to use code execution in an interactive chat with history using the Gemini API.

You can use `client.chats.create` to create a chat session and passes in the code execution tool, enabling the model to generate and run code:

In [11]:
chat = client.chats.create(
    model=MODEL_ID,
    config=GenerateContentConfig(
        tools=[code_execution_tool],
    ),
)

You'll start the chat by asking the model to generate sample time series data with noise and then output a sample of 10 data points:

In [12]:
PROMPT = """Generate and run code to create sample time series data of temperature vs. time in a test furnace.
Add noise to the data. Output a sample of 10 data points from the time series data."""

response = chat.send_message(PROMPT)

Now you can iterate through the response to display any generated Python code and execution results by checking for `part.executable_code` and `part.code_execution_result` in the response parts:

In [13]:
for part in response.candidates[0].content.parts:
    if part.executable_code:
        display(
            Markdown(
                f"""
```py
{part.executable_code.code}
```
"""
            )
        )
    if part.code_execution_result:
        display(Markdown(f"`{part.code_execution_result.output}`"))
        print("\nOutcome:", part.code_execution_result.outcome)


```py
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# Parameters
time_minutes = np.arange(0, 121, 1)  # 0 to 120 minutes
ambient_temp = 25.0
target_temp = 800.0
k = 0.05  # Heating rate constant

# Generate ideal temperature data (Newton's Law of Heating/Cooling approach)
# T(t) = T_ambient + (T_target - T_ambient) * (1 - exp(-k * t))
ideal_temp = ambient_temp + (target_temp - ambient_temp) * (1 - np.exp(-k * time_minutes))

# Add Gaussian noise
noise_std = 2.5
noise = np.random.normal(0, noise_std, size=len(time_minutes))
actual_temp = ideal_temp + noise

# Create DataFrame
df = pd.DataFrame({
    'Time (min)': time_minutes,
    'Temperature (C)': actual_temp
})

# Output a sample of 10 data points
sample_data = df.head(10)
print(sample_data)

```


`   Time (min)  Temperature (C)
0           0        26.241785
1           1        62.451535
2           2       100.370222
3           3       136.758893
4           4       164.898283
5           5       195.844051
6           6       229.813911
7           7       255.785317
8           8       279.328278
9           9       307.194583
`


Outcome: Outcome.OUTCOME_OK


Now you can ask the model to add a smoothed data series to the time series data:

In [14]:
PROMPT = "Rewrite the code to add a data series that smooths the sample data."

response = chat.send_message(PROMPT)

And then display the generated Python code and execution results:

In [15]:
for part in response.candidates[0].content.parts:
    if part.executable_code:
        display(
            Markdown(
                f"""
```py
{part.executable_code.code}
```
"""
            )
        )
    if part.code_execution_result:
        display(Markdown(f"`{part.code_execution_result.output}`"))
        print("\nOutcome:", part.code_execution_result.outcome)


```py
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# 1. Generate Original Data (same as before)
time_minutes = np.arange(0, 121, 1)
ambient_temp = 25.0
target_temp = 800.0
k = 0.05
ideal_temp = ambient_temp + (target_temp - ambient_temp) * (1 - np.exp(-k * time_minutes))
noise = np.random.normal(0, 2.5, size=len(time_minutes))
actual_temp = ideal_temp + noise

# Create DataFrame
df = pd.DataFrame({
    'Time (min)': time_minutes,
    'Temperature (C)': actual_temp
})

# 2. Add Smoothed Data Series
# We use a Simple Moving Average (SMA) with a window of 5 minutes
# min_periods=1 ensures we get values even at the start of the series
df['Smoothed Temp (C)'] = df['Temperature (C)'].rolling(window=5, min_periods=1, center=True).mean()

# Output a sample of the first 10 data points
print(df.head(10))

```


`   Time (min)  Temperature (C)  Smoothed Temp (C)
0           0        26.241785          63.021181
1           1        62.451535          81.455609
2           2       100.370222          98.144144
3           3       136.758893         132.064597
4           4       164.898283         165.537072
5           5       195.844051         196.620091
6           6       229.813911         225.133968
7           7       255.785317         253.593228
8           8       279.328278         280.180457
9           9       307.194583         304.557589
`


Outcome: Outcome.OUTCOME_OK


Finally, you can ask the model to generate descriptive statistics for the time series data:

In [16]:
PROMPT = "Rewrite the code to generate and output descriptive statistics on the time series data."

response = chat.send_message(PROMPT)

And then display the generated Python code and execution results:

In [17]:
for part in response.candidates[0].content.parts:
    if part.executable_code:
        display(
            Markdown(
                f"""
```py
{part.executable_code.code}
```
"""
            )
        )
    if part.code_execution_result:
        display(Markdown(f"`{part.code_execution_result.output}`"))
        print("\nOutcome:", part.code_execution_result.outcome)


```py
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# 1. Generate Data
time_minutes = np.arange(0, 121, 1)
ambient_temp = 25.0
target_temp = 800.0
k = 0.05
ideal_temp = ambient_temp + (target_temp - ambient_temp) * (1 - np.exp(-k * time_minutes))
noise = np.random.normal(0, 2.5, size=len(time_minutes))
actual_temp = ideal_temp + noise

# Create DataFrame
df = pd.DataFrame({
    'Time (min)': time_minutes,
    'Temperature (C)': actual_temp
})

# 2. Add Smoothed Data Series
df['Smoothed Temp (C)'] = df['Temperature (C)'].rolling(window=5, min_periods=1, center=True).mean()

# 3. Generate Descriptive Statistics
# We exclude 'Time' from the statistics as it is a linear index
stats = df[['Temperature (C)', 'Smoothed Temp (C)']].describe()

# Adding variance specifically as it's useful for noise analysis
variance = df[['Temperature (C)', 'Smoothed Temp (C)']].var().to_frame(name='variance').T
stats = pd.concat([stats, variance])

print("Descriptive Statistics for Furnace Temperature Data:")
print(stats)

```


`Descriptive Statistics for Furnace Temperature Data:
          Temperature (C)  Smoothed Temp (C)
count          121.000000         121.000000
mean           668.801372         668.953269
std            187.844672         186.708358
min             26.241785          63.021181
25%            625.569859         626.813876
50%            761.875591         761.086311
75%            791.239664         791.612873
max            803.432030         800.216947
variance     35285.620817       34860.010924
`


Outcome: Outcome.OUTCOME_OK


This chat example demonstrates how you can use the Gemini API with code execution as a powerful tool for exploratory data analysis and more. Go forth and adapt this approach to your own projects and use cases!

### Code execution in a streaming session

You can also use the code execution functionality with streaming output from the Gemini API.

The following code demonstrates how the Gemini API can generate and execute code while streaming the results:

In [18]:
PROMPT = """Generate and run code to create a list of 20 random names, then
create a new list with just the names containing the letter 'a', then output the
number of names that contain 'a', and finally show me that new list."""

for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[code_execution_tool],
    ),
):
    if chunk.candidates and chunk.candidates[0].content:
        if chunk.candidates[0].content.parts is not None:
            for part in chunk.candidates[0].content.parts:
                if part.text:
                    display(Markdown("#### Natural language stream"))
                    display(Markdown(part.text))
                    display(Markdown("---"))
                if part.executable_code:
                    display(Markdown("#### Code stream"))
                    display(
                        Markdown(
                            f"""
```py
{part.executable_code.code}
```
"""
                        )
                    )
                    display(Markdown("---"))
                if part.code_execution_result:
                    display(Markdown("#### Code result"))
                    display(
                        Markdown(
                            f"""
    ```
    {part.code_execution_result.output}
    ```
    """
                        )
                    )
                    display(Markdown("---"))

#### Code stream


    ```py
    import random

# A list of common names to sample from
all_names = [
    "James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda",
    "David", "Elizabeth", "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica",
    "Thomas", "Sarah", "Christopher", "Karen", "Charles", "Lisa", "Daniel", "Nancy",
    "Matthew", "Betty", "Anthony", "Sandra", "Mark", "Margaret", "Donald", "Ashley",
    "Steven", "Kimberly", "Paul", "Emily", "Andrew", "Donna", "Joshua", "Michelle",
    "Kenneth", "Dorothy", "Kevin", "Carol", "Brian", "Amanda", "George", "Melissa",
    "Timothy", "Deborah"
]

# 1. Create a list of 20 random names
random_names = random.sample(all_names, 20)

# 2. Create a new list with just the names containing the letter 'a' (case-insensitive)
names_with_a = [name for name in random_names if 'a' in name.lower()]

# 3. Output the number of names that contain 'a'
count_a = len(names_with_a)

print(f"Randomly selected 20 names: {random_names}")
print(f"\nNumber of names containing 'a': {count_a}")
print(f"Names containing 'a': {names_with_a}")

    ```
    

---

#### Code result


    ```
    Randomly selected 20 names: ['Nancy', 'Michael', 'Jennifer', 'James', 'Brian', 'Susan', 'Donna', 'George', 'Kenneth', 'Elizabeth', 'Michelle', 'Andrew', 'Kevin', 'Daniel', 'Matthew', 'Steven', 'Joshua', 'Lisa', 'Sarah', 'David']

Number of names containing 'a': 14
Names containing 'a': ['Nancy', 'Michael', 'James', 'Brian', 'Susan', 'Donna', 'Elizabeth', 'Andrew', 'Daniel', 'Matthew', 'Joshua', 'Lisa', 'Sarah', 'David']

    ```
    

...

. Daniel
10. Matthew
11. Joshua
12. Lisa
13. Sarah
14. David

**Full list of 20 random names generated:**
`['Nancy', 'Michael', 'Jennifer', 'James', 'Brian', 'Susan', 'Donna', 'George',

---

#### Natural language stream

 'Kenneth', 'Elizabeth', 'Michelle', 'Andrew', 'Kevin', 'Daniel', 'Matthew', 'Steven', 'Joshua', 'Lisa', 'Sarah', 'David']`

---

This streaming example demonstrated how the Gemini API can generate, execute code, and provide results within a streaming session.

## Summary

Refer to the [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/code-execution) for more details about code execution, and in particular, the [recommendations](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/code-execution#code-execution-vs-function-calling) regarding differences between code execution and [function calling](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/function-calling).

### Next steps

- See the [Google Gen AI SDK reference docs](https://googleapis.github.io/python-genai/)
- Explore other notebooks in the [Google Cloud Generative AI GitHub repository](https://github.com/GoogleCloudPlatform/generative-ai)
- Explore AI models in [Model Garden](https://cloud.google.com/vertex-ai/generative-ai/docs/model-garden/explore-models)